In [1]:
#Install libraries

!pip install torch
!pip install dice-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.2 MB/s eta 0:00:00


In [2]:
#Import libraries

from torch.utils.data import DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [3]:
# %% Custom Dataloader
class CustomDataLoader:
  def __init__(self, filepath):
    self.filepath=filepath
    self.data=None

  def load_dataset(self):
    self.data=pd.read_csv(self.filepath)

  def preprocess_data(self):
    #Implement preprocessing here
    self.data.dropna(inplace=True)
    # Ensure get_dummies directly produces integer columns (0 or 1)
    # This makes data_loader.data consistent with integer-encoded booleans.
    self.data=pd.get_dummies(self.data, dtype=int)


  def get_data_split(self, test_size=0.2, random_state=42):
    X=self.data.drop('stroke', axis=1)
    y=self.data['stroke']
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

  def oversample(self, X_train, y_train):
    smote=SMOTE(random_state=42)
    X_res, y_res=smote.fit_resample(X_train, y_train)
    return X_res, y_res

# %%Load and preprocess data

data_loader=CustomDataLoader('/content/drive/MyDrive/imgdata/healthcare-dataset-stroke-data.csv')
data_loader.load_dataset()
data_loader.preprocess_data()

Train_Test data split

In [4]:
#Split the data for evaluation

X_train, X_test, y_train, y_test = data_loader.get_data_split()
#Oversample the train data
X_train, y_train = data_loader.oversample(X_train, y_train)
print(X_train.shape)
print(X_test.shape)

(7542, 22)
(982, 22)


In [5]:
X_test[0:1]

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4336,53802,80.0,0,1,125.32,32.9,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0


In [6]:
#Random Forest Classifier
rf=RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred=rf.predict(X_test)
print(f"F1 Score{f1_score(y_test, y_pred)}")
print(f"Accuracy{accuracy_score(y_test, y_pred)}")

F1 Score0.03389830508474576
Accuracy0.9419551934826884


In [7]:
X_test

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4336,53802,80.0,0,1,125.32,32.9,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0
3709,1454,42.0,0,0,84.03,31.4,1,0,0,1,...,0,1,0,0,0,1,0,0,1,0
964,59336,66.0,1,0,74.90,32.1,0,1,0,0,...,0,1,0,0,1,0,0,0,1,0
2647,66264,29.0,0,0,102.40,26.9,0,1,0,0,...,0,0,0,0,0,1,0,0,0,1
3262,14376,47.0,0,0,88.49,22.2,0,1,0,0,...,0,1,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,60047,22.0,0,0,58.38,36.0,0,1,0,1,...,0,1,0,0,1,0,0,0,1,0
205,51314,78.0,0,0,106.74,33.0,1,0,0,0,...,0,1,0,0,0,1,0,1,0,0
3838,53759,56.0,0,0,122.73,37.5,0,1,0,0,...,0,0,1,0,0,1,0,1,0,0
5010,58635,72.0,0,0,74.17,35.5,1,0,0,0,...,0,0,1,0,0,1,0,1,0,0


Create Counterfactual Explanations

In [8]:
# %% Create diverse counterfactual explanations
import dice_ml

data_dice = dice_ml.Data(dataframe=data_loader.data,
                       # For perturbation strategy
                       continuous_features=['age',
                                           'avg_glucose_level',
                                           'bmi'],
                       outcome_name='stroke')

Creating the Data and Model Objects for DiCE

In [9]:
#Model

# Re-initialize rf_dice and explainer to use the updated data_dice object.
rf_dice=dice_ml.Model(model=rf,
                      #There exist backend for tf, torch
                      backend='sklearn')
explainer=dice_ml.Dice(data_dice,
                       rf_dice,
                       method='random')

Generating and visualizing counterfactual explanations

In [10]:
#%% Create explanation
#generate CF based on the model
# Use .copy() to create an independent DataFrame and prevent SettingWithCopyWarning
input_datapoint=X_test[0:1].copy()

cf=explainer.generate_counterfactuals(input_datapoint,
                                      total_CFs=3,
                                      desired_class='opposite')

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '56324' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is 

In [11]:
X_test[0:1]

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4336,53802,80.0,0,1,125.32,32.9,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0


In [12]:
#Visualize it
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,53802,80.0,0,1,125.32,32.900002,0,1,0,0,...,1,0,0,1,0,1,0,0,0,0



Diverse Counterfactual set (new outcome: 1)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,0.0,-,-,-,-,-,1.0
1,-,-,-,-,-,-,-,-,-,-,...,-,-,-,0.0,-,-,-,-,-,1.0
2,-,-,-,-,-,-,-,0.0,-,-,...,-,-,-,-,-,-,-,-,-,-


In [13]:
#Creating Feasible(conditional) Counterfactuals
features_to_vary=['avg_glucose_level',
                  'bmi',
                  'smoking_status_smokes']
permitted_range={'avg_glucose_level':[10,300],
                 'bmi':[15,45]}

i=613
input_datapoint2=X_test[i:i+1]

print(input_datapoint2.to_string(index=False))


   id  age  hypertension  heart_disease  avg_glucose_level  bmi  gender_Female  gender_Male  gender_Other  ever_married_No  ever_married_Yes  work_type_Govt_job  work_type_Never_worked  work_type_Private  work_type_Self-employed  work_type_children  Residence_type_Rural  Residence_type_Urban  smoking_status_Unknown  smoking_status_formerly smoked  smoking_status_never smoked  smoking_status_smokes
67981 66.0             0              0             151.16 27.5              0            1             0                0                 1                   0                       0                  1                        0                   0                     0                     1                       0                               1                            0                      0


In [14]:
#Now generating explanations using the new feature weights

cf=explainer.generate_counterfactuals(input_datapoint2,
                                      total_CFs=3,
                                      desired_class='opposite',
                                      permitted_range=permitted_range,
                                      features_to_vary=features_to_vary)
#Visualize it
cf.visualize_as_dataframe(show_only_changes=True)

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

Query instance (original outcome : 1)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,67981,66.0,0,0,151.160004,27.5,0,1,0,0,...,1,0,0,0,1,0,1,0,0,1



Diverse Counterfactual set (new outcome: 0)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,-,97.87,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0
1,-,-,-,-,-,16.4,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0
2,-,-,-,-,253.85,31.9,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0


In [16]:
#Note:
#If you don't constraint the predictions values than it might generate some random crazy values